In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

        # import numpy as np # linear algebra
        # import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

        # import os
        # for dirname, _, filenames in os.walk('/kaggle/input'):
        #     for filename in filenames:
        #         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

        # import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Beginning from here.

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Find Your Kaggle Path ---
# In Kaggle, datasets are usually mounted in /kaggle/input/
# UPDATE "your-dataset-name" to whatever you named your upload!
# Set the path to your data_processed directory on your local machine (or Kaggle input path)
DATA_ROOT = r"<PATH_TO_DATA_PROCESSED_DIR>"  # e.g., r"./data_processed" (local) or "/kaggle/input/<dataset-name>/data_processed" (Kaggle)
IMAGE_DIR = os.path.join(DATA_ROOT, "sar_tiles")
MASK_DIR = os.path.join(DATA_ROOT, "masks")

# --- 2. Define the Custom PyTorch Dataset ---
class CoastlineDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        # Sort them to ensure image_001 aligns exactly with mask_001
        self.images = sorted(os.listdir(image_dir))
        self.masks = sorted(os.listdir(mask_dir))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        # Load as Grayscale (1 channel)
        image = Image.open(img_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        # Convert Image to PyTorch Tensor (scales pixels to 0.0 - 1.0)
        if self.transform:
            image_tensor = self.transform(image)
            
        # Convert Mask to Tensor and force strict 0 and 1 binary labels
        mask_tensor = transforms.ToTensor()(mask)
        mask_tensor = (mask_tensor > 0.5).float() 

        return image_tensor, mask_tensor

# --- 3. Initialize the DataLoader ---
# Standard image transformation (just converting to Tensor for now)
transform = transforms.Compose([
    transforms.ToTensor()
])

# Create the dataset object
dataset = CoastlineDataset(IMAGE_DIR, MASK_DIR, transform=transform)

# DataLoader acts as the conveyor belt, serving 8 images at a time
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

print(f"Total tiles loaded: {len(dataset)}")
print(f"Total batches per epoch: {len(dataloader)}")

# --- 4. Smoke Test: Fetch One Batch ---
images, masks = next(iter(dataloader))
print(f"Image Batch Shape: {images.shape} (Batch, Channel, Height, Width)")
print(f"Mask Batch Shape: {masks.shape}")

# Visualize the first image and mask in the batch
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# Tensors are (C, H, W), so we permute to (H, W, C) for matplotlib
axes[0].imshow(images[0].permute(1, 2, 0).numpy(), cmap='gray')
axes[0].set_title("PyTorch Tensor: SAR Image")
axes[1].imshow(masks[0].permute(1, 2, 0).numpy(), cmap='gray')
axes[1].set_title("PyTorch Tensor: Binary Mask")
# plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__() # old Python 2.0 syntax; just the same as 'super().__init__()'
        
        # --- ENCODER (Downsampling) ---
        # Input: 1 channel (Grayscale SAR), Output: 64 features
        self.enc_conv1 = self.double_conv(1, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.enc_conv2 = self.double_conv(64, 128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.enc_conv3 = self.double_conv(128, 256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # --- BOTTLENECK ---
        self.bottleneck_conv = self.double_conv(256, 512)
        
        # --- DECODER (Upsampling) ---
        self.up_trans1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec_conv1 = self.double_conv(512, 256) # 512 because 256 from up + 256 skip connection
        
        self.up_trans2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec_conv2 = self.double_conv(256, 128)
        
        self.up_trans3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec_conv3 = self.double_conv(128, 64)
        
        # --- FINAL OUTPUT ---
        # Compresses the 64 features down to 1 single channel (the binary mask)
        self.out_conv = nn.Conv2d(64, 1, kernel_size=1)

    def double_conv(self, in_channels, out_channels):
        """Helper function: A standard Convolution -> BatchNorm -> ReLU block applied twice."""
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # 1. Encoder Path
        e1 = self.enc_conv1(x)
        e2 = self.enc_conv2(self.pool1(e1))
        e3 = self.enc_conv3(self.pool2(e2))
        
        # 2. Bottleneck
        b = self.bottleneck_conv(self.pool3(e3))
        
        # 3. Decoder Path (with Skip Connections)
        d1 = self.up_trans1(b)
        d1 = torch.cat([e3, d1], dim=1) # The Skip Connection!
        d1 = self.dec_conv1(d1)
        
        d2 = self.up_trans2(d1)
        d2 = torch.cat([e2, d2], dim=1) # The Skip Connection!
        d2 = self.dec_conv2(d2)
        
        d3 = self.up_trans3(d2)
        d3 = torch.cat([e1, d3], dim=1) # The Skip Connection!
        d3 = self.dec_conv3(d3)
        
        # 4. Final Output (Sigmoid forces the output between 0 and 1)
        out = torch.sigmoid(self.out_conv(d3))
        return out

# --- CPU Smoke Test ---
print("Initializing U-Net...")
model = UNet()

# Let's pass that batch of 8 images we loaded earlier through the untrained network!
print(f"Passing a batch of shape {images.shape} through the network...")
output_masks = model(images)

print(f"Network Output Shape: {output_masks.shape}")
print("SUCCESS: The architecture is mathematically sound!")

In [ ]:
import torch.optim as optim
import torch.nn as nn

# 1. Move the model from the CPU to the GPU
# 'cuda' is NVIDIA's parallel computing platform
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model = UNet().to(device)

# 2. Setup the Grader and the Learner
# BCELoss = Binary Cross Entropy (Perfect for comparing 1s and 0s)
criterion = nn.BCELoss() 

# Adam is the industry standard optimizer. lr is the Learning Rate (how aggressively it turns the dials)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# 3. The 50-Epoch Training Loop
num_epochs = 150
print(f"Starting the {num_epochs}-Epoch Full Burn...")

for epoch in range(num_epochs):
    # Set the model to training mode (enables gradient tracking)
    model.train() 
    epoch_loss = 0.0
    
    # Loop through the dataset, 8 images at a time
    for batch_idx, (images, masks) in enumerate(dataloader):
        
        # Ship the images and answer keys to the GPU memory
        images = images.to(device)
        masks = masks.to(device)
        
        # Step A: Zero out the gradients from the last batch
        optimizer.zero_grad()
        
        # Step B: Forward Pass (The model makes a guess)
        outputs = model(images)
        
        # Step C: Calculate the Loss (Grade the guess)
        loss = criterion(outputs, masks)
        
        # Step D: Backward Pass (Calculate how to fix the dials)
        loss.backward()
        
        # Step E: Update the Weights (Actually turn the dials)
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Print an update every 10 batches so we know it hasn't frozen
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")
            
    # Print average loss for the whole epoch
    avg_loss = epoch_loss / len(dataloader)
    print(f"--- Epoch {epoch+1} Completed | Average Loss: {avg_loss:.4f} ---")

print("\nSUCCESS! The U-Net has finished training.")

# 4. Save the Model Weights
# We save the 'state_dict' (the learned weights and biases), not the whole model structure.
# This is the PyTorch best practice.
save_path = r"<PATH_TO_SAVE_MODEL_WEIGHTS>"  # e.g., r"./unet_coastline_weights_sb.pth"
torch.save(model.state_dict(), save_path)
print(f"Model saved successfully to: {save_path}")